<a href="https://colab.research.google.com/github/liamscanlon5/DS_Analytics_Demand_Estimation_FinalProject/blob/main/DataAnalystNotebook_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Demand Estimation and Market Analysis: Air Fryers

In this lab, you will study the market for air fryers using brand-year data aggregated from Amazon purchases. The goal is to move from descriptive analysis to a simple demand model, and then use that model to infer markups and unit costs.

Use the cleaned file:

```python
air_fryers_clean_brand_year.csv
```

This file keeps the top 10 air-fryer brands from 2019-2023 and drops the long tail of very small brands. The variable `brand_share` has already been recomputed within this cleaned name-brand market, so shares sum to 1 within each year.

## Data

Each row is one brand in one year.

Important columns:

- `year`: calendar year
- `brand`: air-fryer brand
- `purchase_count`: number of purchases by that brand in that year
- `product_count`: number of distinct products observed for that brand-year
- `avg_price`: average price for that brand-year
- `avg_rating`: average review rating for that brand-year
- `brand_share`: purchase share within the cleaned air-fryer market in that year
- `log_brand_share`: `np.log(brand_share)`, already computed for convenience
- `compact_share`, `dual_basket_share`, `oven_style_share`, `rotisserie_share`, `window_share`: product characteristic shares for the brand-year

The original lecture wrote the demand equation using an outside option:

$$
\log(s_{bt}) - \log(s_{ot}).
$$

For this cleaned dataset, we dropped the nuisance long-tail brands instead of treating them as an outside option. You should therefore use:

$$
y_{bt} = \log(s_{bt})
$$

as the outcome and include **year dummies**. The year dummies absorb the year-specific denominator of the multinomial logit share equation. This keeps the assignment focused on the cleaned name-brand market.

## 1. Data Analysis

Load `air_fryers_clean_brand_year.csv`.

1. Verify that the data contain 10 brands and the years 2019-2023.
2. Plot the following over time by brand:
   - average price
   - average rating
   - brand market share
3. Summarize the product characteristics:
   - Which features are common?
   - Which features are rare?
   - Are there brands that seem to specialize in different product types?
4. Write a short paragraph describing the market. Which brands are expensive? Which brands have large shares? Does the market look stable over time?

This part of the work is the **data analyst** role: making the data trustworthy, visual, and interpretable before building a model.

In [ ]:
from google.colab import files
#this shows a picker to upload the air fryers dataset
uploaded = files.upload()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression

df = pd.read_csv('air_fryers_clean_brand_year.csv')

print(df.shape)
print(df.head())

# 1. Verify the data contains 10 brands and the yrs 19-23
num_brands = df['brand'].nunique()
years = df['year'].unique()

print(f"Number of distinct brands: {num_brands}")
print(f"Years in dataset: {sorted(years)}")

In [ ]:
# 2. Plots by Brand

fig, axes = plt.subplots(3, 1, figsize=(10, 7.5), sharex=True)
sns.set_theme(style="whitegrid")

colors = sns.color_palette("tab10", n_colors=10)

# Avg price over time Plot
sns.lineplot(data=df, x='year', y='avg_price', hue='brand', palette=colors,
             marker='o', ax=axes[0], legend=False, alpha=0.8)
axes[0].set_title('Average Price Over Time by Brand', fontsize=12)
axes[0].set_ylabel('Average Price ($)')

# Avg rating over time Plot
sns.lineplot(data=df, x='year', y='avg_rating', hue='brand', palette=colors,
             marker='o', ax=axes[1], legend=False, alpha=0.8)
axes[1].set_title('Average Rating Over Time by Brand', fontsize=12)
axes[1].set_ylabel('Average Rating')

# Brand Mkt share over time Plot
sns.lineplot(data=df, x='year', y='brand_share', hue='brand', palette=colors,
             marker='o', ax=axes[2], alpha=0.8)
axes[2].set_title('Market Share Over Time by Brand', fontsize=12)
axes[2].set_ylabel('Market Share')
axes[2].set_xticks(range(2019, 2024))


axes[2].legend(bbox_to_anchor=(1.05, 1.5), loc='center left', borderaxespad=0., title='Brand')

plt.subplots_adjust(hspace=0.15, right=0.85)

# Extend the x-axis slightly
axes[2].set_xlim(2018.8, 2023.2)

plt.show()

3. Which features are common? Which features are rare? Are there brands that seem to specialize in different products?

In [ ]:
# make list of important col names so we can just grab them from the df
feature_cols = [
    'compact_share',
    'dual_basket_share',
    'oven_style_share',
    'rotisserie_share',
    'window_share',
]

# Product characteristic summaries.
feature_summary = (
    df[feature_cols]
    .mean()
    .sort_values(ascending=False)
    .to_frame('average_share')
)

brand_feature_summary = df.groupby('brand')[feature_cols].mean().sort_index()
brand_market_summary = (
    df.groupby('brand')
    .agg(
        avg_price=('avg_price', 'mean'),
        avg_rating=('avg_rating', 'mean'),
        avg_brand_share=('brand_share', 'mean'),
        min_brand_share=('brand_share', 'min'),
        max_brand_share=('brand_share', 'max'),
    )
    .sort_values('avg_brand_share', ascending=False)
)

print('Average feature shares across all brand-years')
display(feature_summary)
print('Average product-characteristic mix by brand')
display(brand_feature_summary)
print('Average market position by brand')
display(brand_market_summary)

fig, ax = plt.subplots(figsize=(9, 5))
sns.heatmap(brand_feature_summary, annot=True, cmap='Blues', fmt='.2f', ax=ax)
ax.set_title('Average Feature Share by Brand')
ax.set_xlabel('Feature')
ax.set_ylabel('Brand')
plt.tight_layout()
plt.show()

The most common characteristic is compact_share, which almost all airfryers are classified as, about 0.98 on average.  oven_style_share is also common, at about 0.56.  rotisserie, window, and dual_basket are pretty rare in the dataset.  Brands do seem to specialize somewhat.  For example most brands just do compact/oven_style.  Chefman is more diversified and has a decent share of rostisserie and window.  

4. Write a short paragraph describing the market. Which brands are expensive? Which brands have large shares? Does the market look stable over time?


Cusinart and Oster are the most expensive brands.  Ninja and InstantPot have the largest shares, but the market is not super stable.  For example, GoWise had the largest share year out of all of them at 29%, but it's worst year was 5%.  Also worth taking note, higher price doesn't neccessarily mean higher share, because the two most expensive brands combine to only about an 11% market share.